# SNLI exploration with Polars\n
\n
This notebook loads SNLI JSONL files and performs quick EDA with Polars.

In [2]:
from pathlib import Path
import polars as pl

In [4]:
data_dir = Path('../data/raw/snli')
train_path = data_dir / 'snli_1.0_train.jsonl'
dev_path = data_dir / 'snli_1.0_dev.jsonl'
test_path = data_dir / 'snli_1.0_test.jsonl'

In [5]:
for p in [train_path, dev_path, test_path]:
    print(p, 'exists:', p.exists())

../data/raw/snli/snli_1.0_train.jsonl exists: True
../data/raw/snli/snli_1.0_dev.jsonl exists: True
../data/raw/snli/snli_1.0_test.jsonl exists: True


In [7]:
train = pl.read_ndjson(train_path)
dev = pl.read_ndjson(dev_path)
test = pl.read_ndjson(test_path)
train.shape, dev.shape, test.shape

((550152, 10), (10000, 10), (10000, 10))

In [8]:
train.head(5)

annotator_labels,captionID,gold_label,pairID,sentence1,sentence1_binary_parse,sentence1_parse,sentence2,sentence2_binary_parse,sentence2_parse
list[str],str,str,str,str,str,str,str,str,str
"[""neutral""]","""3416050480.jpg#4""","""neutral""","""3416050480.jpg#4r1n""","""A person on a horse jumps over…","""( ( ( A person ) ( on ( a hors…","""(ROOT (S (NP (NP (DT A) (NN pe…","""A person is training his horse…","""( ( A person ) ( ( is ( ( trai…","""(ROOT (S (NP (DT A) (NN person…"
"[""contradiction""]","""3416050480.jpg#4""","""contradiction""","""3416050480.jpg#4r1c""","""A person on a horse jumps over…","""( ( ( A person ) ( on ( a hors…","""(ROOT (S (NP (NP (DT A) (NN pe…","""A person is at a diner, orderi…","""( ( A person ) ( ( ( ( is ( at…","""(ROOT (S (NP (DT A) (NN person…"
"[""entailment""]","""3416050480.jpg#4""","""entailment""","""3416050480.jpg#4r1e""","""A person on a horse jumps over…","""( ( ( A person ) ( on ( a hors…","""(ROOT (S (NP (NP (DT A) (NN pe…","""A person is outdoors, on a hor…","""( ( A person ) ( ( ( ( is outd…","""(ROOT (S (NP (DT A) (NN person…"
"[""neutral""]","""2267923837.jpg#2""","""neutral""","""2267923837.jpg#2r1n""","""Children smiling and waving at…","""( Children ( ( ( smiling and )…","""(ROOT (NP (S (NP (NNP Children…","""They are smiling at their pare…","""( They ( are ( smiling ( at ( …","""(ROOT (S (NP (PRP They)) (VP (…"
"[""entailment""]","""2267923837.jpg#2""","""entailment""","""2267923837.jpg#2r1e""","""Children smiling and waving at…","""( Children ( ( ( smiling and )…","""(ROOT (NP (S (NP (NNP Children…","""There are children present""","""( There ( ( are children ) pre…","""(ROOT (S (NP (EX There)) (VP (…"


In [9]:
train.select(pl.all().count()).transpose(
    include_header=True, header_name="column", column_names=["non_null_count"]
)

column,non_null_count
str,u32
"""annotator_labels""",550152
"""captionID""",550152
"""gold_label""",550152
"""pairID""",550152
"""sentence1""",550152
"""sentence1_binary_parse""",550152
"""sentence1_parse""",550152
"""sentence2""",550152
"""sentence2_binary_parse""",550152


In [10]:
for name, df in [('train', train), ('dev', dev), ('test', test)]:
    print(f'\n{name} label distribution:')
    print(df.group_by('gold_label').len().sort('len', descending=True))


train label distribution:
shape: (4, 2)
┌───────────────┬────────┐
│ gold_label    ┆ len    │
│ ---           ┆ ---    │
│ str           ┆ u32    │
╞═══════════════╪════════╡
│ entailment    ┆ 183416 │
│ contradiction ┆ 183187 │
│ neutral       ┆ 182764 │
│ -             ┆ 785    │
└───────────────┴────────┘

dev label distribution:
shape: (4, 2)
┌───────────────┬──────┐
│ gold_label    ┆ len  │
│ ---           ┆ ---  │
│ str           ┆ u32  │
╞═══════════════╪══════╡
│ entailment    ┆ 3329 │
│ contradiction ┆ 3278 │
│ neutral       ┆ 3235 │
│ -             ┆ 158  │
└───────────────┴──────┘

test label distribution:
shape: (4, 2)
┌───────────────┬──────┐
│ gold_label    ┆ len  │
│ ---           ┆ ---  │
│ str           ┆ u32  │
╞═══════════════╪══════╡
│ entailment    ┆ 3368 │
│ contradiction ┆ 3237 │
│ neutral       ┆ 3219 │
│ -             ┆ 176  │
└───────────────┴──────┘


In [12]:
valid = ['entailment', 'contradiction', 'neutral']
train_clean = train.filter(pl.col('gold_label').is_in(valid))
dev_clean = dev.filter(pl.col('gold_label').is_in(valid))
test_clean = test.filter(pl.col('gold_label').is_in(valid))
train_clean.shape, dev_clean.shape, test_clean.shape

((549367, 10), (9842, 10), (9824, 10))

In [14]:
train_clean.select(
    pl.col('sentence1').str.len_chars().mean().alias('premise_avg_chars'),
    pl.col('sentence2').str.len_chars().mean().alias('hypothesis_avg_chars')
)

premise_avg_chars,hypothesis_avg_chars
f64,f64
66.274052,37.473993


Check observations whose labels are "-"

In [ ]:
train.filter(pl.col("gold_label") == "-").head(10)

annotator_labels,captionID,gold_label,pairID,sentence1,sentence1_binary_parse,sentence1_parse,sentence2,sentence2_binary_parse,sentence2_parse
list[str],str,str,str,str,str,str,str,str,str
"[""contradiction"", ""contradiction"", … ""neutral""]","""2677109430.jpg#2""","""-""","""2677109430.jpg#2r1c""","""A small group of church-goers …","""( ( ( A ( small group ) ) ( of…","""(ROOT (S (NP (NP (DT A) (JJ sm…","""A choir performs in front of p…","""( ( A choir ) ( ( performs ( i…","""(ROOT (S (NP (DT A) (NN choir)…"
"[""entailment"", ""entailment"", … ""contradiction""]","""226630666.jpg#1""","""-""","""226630666.jpg#1r1e""","""A woman wearing a pink hat is …","""( ( ( A woman ) ( wearing ( a …","""(ROOT (S (NP (NP (DT A) (NN wo…","""The woman is wearing clothes.""","""( ( The woman ) ( ( is ( weari…","""(ROOT (S (NP (DT The) (NN woma…"
"[""neutral"", ""contradiction"", … ""neutral""]","""179172576.jpg#3""","""-""","""179172576.jpg#3r1n""","""man in red canada shirt standi…","""( man ( in ( ( red ( canada sh…","""(ROOT (NP (NP (NN man)) (PP (I…","""Man standing with three men in…","""( ( Man standing ) ( with ( ( …","""(ROOT (NP (NP (NN Man) (NN sta…"
"[""neutral"", ""contradiction"", … ""entailment""]","""4926115712.jpg#0""","""-""","""4926115712.jpg#0r1n""","""A man in a white jacket standi…","""( ( ( ( A man ) ( in ( a ( whi…","""(ROOT (NP (NP (NP (DT A) (NN m…","""The man was playing crochet wi…","""( ( The man ) ( ( was ( ( play…","""(ROOT (S (NP (DT The) (NN man)…"
"[""neutral"", ""neutral"", … ""contradiction""]","""7249180494.jpg#2""","""-""","""7249180494.jpg#2r1n""","""A swimmer's hand is taken as h…","""( ( ( A ( swimmer 's ) ) hand …","""(ROOT (S (NP (NP (DT A) (NN sw…","""The swimmer is female.""","""( ( The swimmer ) ( ( is femal…","""(ROOT (S (NP (DT The) (NN swim…"
"[""neutral"", ""entailment"", … ""contradiction""]","""6044840724.jpg#4""","""-""","""6044840724.jpg#4r1n""","""People are in a laundry mat wa…","""( People ( ( are ( in ( ( a ( …","""(ROOT (S (NP (NNS People)) (VP…","""Machines are cleaning clothes …","""( Machines ( ( are ( ( cleanin…","""(ROOT (S (NP (NNPS Machines)) …"
"[""neutral"", ""entailment"", … ""contradiction""]","""481632457.jpg#1""","""-""","""481632457.jpg#1r1n""","""An adult Australian Shepherd f…","""( ( An ( adult ( Australian Sh…","""(ROOT (S (NP (DT An) (JJ adult…","""A puppy is running towards its…","""( ( A puppy ) ( ( is ( running…","""(ROOT (S (NP (DT A) (NN puppy)…"
"[""entailment"", ""neutral"", … ""neutral""]","""2770821714.jpg#1""","""-""","""2770821714.jpg#1r1e""","""Two adults and a child are sta…","""( ( ( ( Two adults ) and ) ( a…","""(ROOT (S (NP (NP (CD Two) (NNS…","""People stand on a stone in fro…","""( People ( ( stand ( on ( ( a …","""(ROOT (S (NP (NNS People)) (VP…"
"[""contradiction"", ""neutral"", … ""contradiction""]","""4274395288.jpg#4""","""-""","""4274395288.jpg#4r1c""","""There is a large hole in the r…","""( There ( ( is ( ( a ( large h…","""(ROOT (S (NP (EX There)) (VP (…","""The large hole in the yard is …","""( ( ( The ( large hole ) ) ( i…","""(ROOT (S (NP (NP (DT The) (JJ …"


In [20]:
label_dash_train_sample = train.filter(pl.col("gold_label") == "-").head(10)

label_dash_train_sample.select(
    pl.col("sentence1"),
    pl.col("sentence2"),
)

sentence1,sentence2
str,str
"""A small group of church-goers …","""A choir performs in front of p…"
"""A woman wearing a pink hat is …","""The woman is wearing clothes."""
"""man in red canada shirt standi…","""Man standing with three men in…"
"""A man in a white jacket standi…","""The man was playing crochet wi…"
"""A swimmer's hand is taken as h…","""The swimmer is female."""
"""People are in a laundry mat wa…","""Machines are cleaning clothes …"
"""An adult Australian Shepherd f…","""A puppy is running towards its…"
"""Two adults and a child are sta…","""People stand on a stone in fro…"
"""There is a large hole in the r…","""The large hole in the yard is …"


In [21]:
row = label_dash_train_sample.row(0, named=True)

print("Sentence 1:", row["sentence1"])
print("Sentence 2:", row["sentence2"])

Sentence 1: A small group of church-goers watch a choir practice.
Sentence 2: A choir performs in front of packed crowd.


In [25]:
row = label_dash_train_sample.row(0, named=True)

print("gold_label:", row["gold_label"])
print("annotator_labels:", row.get("annotator_labels"))

gold_label: -
annotator_labels: ['contradiction', 'contradiction', 'neutral', 'neutral']


Clearly, the rows with gold label "-" are those for which there was not a label that won over the others.

We shall remove those from our exploration.

In [27]:
train_clean = train.filter(pl.col("gold_label") != "-")
dev_clean = dev.filter(pl.col("gold_label") != "-")
test_clean = test.filter(pl.col("gold_label") != "-")

print("After removing rows with gold_label '-':")
print("Train shape:", train_clean.shape)
print("Dev shape:", dev_clean.shape)
print("Test shape:", test_clean.shape)

After removing rows with gold_label '-':
Train shape: (549367, 10)
Dev shape: (9842, 10)
Test shape: (9824, 10)
